# Legal-RAG: Pipeline Ingestion & Retrieval Test
Notebook này chạy bộ Chunker & Embedder 100% On-Premise và tương tác trực tiếp với Local Docker DB.

In [1]:
import sys
import os

# 1. Trỏ vào thư mục gốc Legal-RAG
sys.path.append(os.path.abspath("d:/iCOMM/Legal-RAG"))

# 2. Ghi đè cấu hình Môi trường thành Cục Bộ (Local Docker)
os.environ["QDRANT_URL"] = "http://localhost:6335"
os.environ["QDRANT_API_KEY"] = "" 

# BỔ SUNG DÒNG NÀY ĐỂ ĐỒNG BỘ TÊN COLLECTION
os.environ["QDRANT_COLLECTION"] = "legal_hybrid_rag_docs"

os.environ["NEO4J_URI"] = "bolt://localhost:7687"
os.environ["NEO4J_USERNAME"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "u7aGQYEWeFJD-jyeHB4ATtoAud73PptW35M1RzFlT-0"

# 3. Setup HF Cache
os.environ["HF_DATASETS_CACHE"] = r"D:\huggingface_cache"

print(f"✅ Môi trường gắn kết thành công tới Root Directory.")

✅ Môi trường gắn kết thành công tới Root Directory.


In [2]:
import pandas as pd
from datasets import load_dataset
from backend.utils.document_parser import DocumentParser
from backend.retrieval.chunker.metadata import normalize_doc_key

# 1. Đọc 100 ID từ file CSV được yêu cầu
csv_path = r"D:\iCOMM\Legal-RAG\metadata_the_thao_y_te_100.csv"
target_ids = set()
if os.path.exists(csv_path):
    df_csv = pd.read_csv(csv_path)
    if "id" in df_csv.columns:
        target_ids = set(df_csv["id"].astype(str).tolist())
        print(f"-> Đã đọc {len(target_ids)} IDs văn bản.")
else:
    print("⚠️ Không tìm thấy file CSV!")

# 2. Tải Dataset Full từ HuggingFace
print("-> Đang tải dataset từ HuggingFace...")
ds_metadata_all = load_dataset("nhn309261/vietnamese-legal-docs", "metadata", split="data")
ds_content_all = load_dataset("nhn309261/vietnamese-legal-docs", "content", split="data")

# 3. Lập chỉ mục O(1) Memory mapping (Tránh bùng nạp RAM)
df_meta = ds_metadata_all.to_pandas()
df_meta["id"] = df_meta["id"].astype(str)
all_meta_records = df_meta.to_dict("records")
print("-> Đang lập chỉ mục O(1)...")
meta_id_to_idx = {row["id"]: idx for idx, row in enumerate(all_meta_records)}
meta_docnum_to_id = {}
for row in all_meta_records:
    doc_num = str(row.get("document_number", ""))
    if doc_num:
        key = normalize_doc_key(doc_num)
        if key:
            meta_docnum_to_id[key] = row["id"]

content_id_to_idx = {str(row["id"]): i for i, row in enumerate(ds_content_all.select_columns(["id"]))}

print(f"✅ Đã tải và lập map {len(all_meta_records)} tài liệu Metadata.")

class DynamicContentLookup(dict):
    def __init__(self, ds_content, content_id_to_idx, ds_meta, meta_docnum_to_id, meta_id_to_idx):
        self.ds_content = ds_content
        self.content_id_to_idx = content_id_to_idx
        self.ds_meta = ds_meta
        self.meta_docnum_to_id = meta_docnum_to_id
        self.meta_id_to_idx = meta_id_to_idx
    def get(self, key, default=None):
        if key in self:
            return super().get(key)
        t_id_str = self.meta_docnum_to_id.get(key)
        if t_id_str:
            idx_cont = self.content_id_to_idx.get(t_id_str)
            if idx_cont is not None:
                text = self.ds_content[idx_cont].get("content", "")
                self[key] = text
                return text
        return default

global_doc_lookup = DynamicContentLookup(ds_content_all, content_id_to_idx, all_meta_records, meta_docnum_to_id, meta_id_to_idx)
meta_by_docnum_lookup = {}


-> Đã đọc 100 IDs văn bản.
-> Đang tải dataset từ HuggingFace...
-> Đang lập chỉ mục O(1)...
✅ Đã tải và lập map 518255 tài liệu Metadata.


In [ ]:
from backend.retrieval.chunker.core import AdvancedLegalChunker
from backend.retrieval.chunker.metadata import normalize_doc_key
from backend.retrieval.chunker import relations as rel
chunker = AdvancedLegalChunker()

all_chunks = []
processed_ids = set()
discovered_depth1_ids = set()  # ID văn bản tham chiếu cần nạp thêm
MAX_RECURSIVE_DOCS = 300
BATCH_SIZE = 10

# ═══════════════════════════════════════════════════
# PHA 1: Chunk toàn bộ văn bản GỐC (depth=0) + Phát hiện Target Docs
# ═══════════════════════════════════════════════════
print("═" * 60)
print("PHA 1: Chunking văn bản GỐC (depth=0) + Discovery Target Docs (BATCH 10)")
print("═" * 60)

target_ids_list = list(target_ids)
for i in range(0, len(target_ids_list), BATCH_SIZE):
    batch_ids = target_ids_list[i : i + BATCH_SIZE]
    batch_docs = []
    batch_meta_map = {}
    
    for doc_id in batch_ids:
        if doc_id in processed_ids: continue
        
        idx_content = content_id_to_idx.get(doc_id)
        idx_meta = meta_id_to_idx.get(doc_id)
        if idx_content is None or idx_meta is None:
            processed_ids.add(doc_id)
            continue
        
        content = ds_content_all[idx_content].get("content", "")
        meta = all_meta_records[idx_meta]
        if not content: 
            processed_ids.add(doc_id)
            continue
            
        doc_num = str(meta.get("document_number", ""))
        batch_docs.append({"source_doc": doc_num, "content": content})
        batch_meta_map[doc_num] = {"id": doc_id, "meta": meta, "content": content}

    if not batch_docs: continue
    
    # BƯỚC 1: TRÍCH XUẤT RELATION THEO MẺ 10 (Cell 3 & 4)
    print(f"  [BATCH] Đang trích xuất quan hệ cho mẻ {len(batch_docs)} văn bản...")
    
    # Bọc Try-Catch để đảm bảo pipeline không bao giờ chết
    try:
        rels_map = rel.extract_ontology_relationships_batch(batch_docs, global_doc_lookup=global_doc_lookup)
    except Exception as e:
        print(f"  ⚠️ LỖI BATCH: Bỏ qua trích xuất quan hệ mẻ này do lỗi LLM - {e}")
        rels_map = {} # Dự phòng rỗng để vẫn chunk được văn bản gốc
    
    # BƯỚC 2: CHUNK TỪNG VĂN BẢN VỚI RELATION ĐÃ CÓ
    for doc_info in batch_docs:
        s_doc = doc_info["source_doc"]
        doc_data = batch_meta_map[s_doc]
        doc_id = doc_data["id"]
        meta = doc_data["meta"]
        content = doc_data["content"]
        
        precomputed = rels_map.get(s_doc, [])
        
        key = normalize_doc_key(s_doc)
        if key: meta_by_docnum_lookup[key] = meta
        
        doc_chunks = chunker.process_document(
            content=content,
            metadata=meta,
            global_doc_lookup=global_doc_lookup,
            precomputed_rels=precomputed
        )
        all_chunks.extend(doc_chunks)
        processed_ids.add(doc_id)

        # Discovery: Phát hiện các văn bản tham chiếu
        for chunk in doc_chunks:
            neo4j_meta = chunk.get("neo4j_metadata", chunk.get("metadata", {}))
            for r in neo4j_meta.get("ontology_relations", []):
                t_key = normalize_doc_key(r.get("target_doc", ""))
                t_id = meta_docnum_to_id.get(t_key)
                if t_id and t_id not in processed_ids and t_id not in target_ids:
                    discovered_depth1_ids.add(t_id)

    print(f"  Đã hoàn thành mẻ {i//BATCH_SIZE + 1}. Tổng processed: {len(processed_ids)}/{len(target_ids)}. Target docs mới: {len(discovered_depth1_ids)}")

# Giới hạn số lượng target docs
if len(discovered_depth1_ids) > MAX_RECURSIVE_DOCS:
    print(f"  ⚠️ Phát hiện {len(discovered_depth1_ids)} target docs, giới hạn còn {MAX_RECURSIVE_DOCS}.")
    discovered_depth1_ids = set(list(discovered_depth1_ids)[:MAX_RECURSIVE_DOCS])

print(f"\n✅ PHA 1 hoàn tất: {len(processed_ids)} văn bản gốc → {len(all_chunks)} chunks.")
print(f"   Phát hiện {len(discovered_depth1_ids)} văn bản tham chiếu cần nạp thêm.")


════════════════════════════════════════════════════════════
PHA 1: Chunking văn bản GỐC (depth=0) + Discovery Target Docs (BATCH 10)
════════════════════════════════════════════════════════════
  [BATCH] Đang trích xuất quan hệ cho mẻ 10 văn bản...
      [Pha 1] 🚀 Tìm thấy 164 đoạn văn. Bắt đầu gọi Batch LLM...
      [Pha 1] Đang xử lý mẻ phụ 1 (Gồm 20 đoạn)...


❌ [InternalLLMClient] Batch failed (Attempt 1/2): HTTPConnectionPool(host='10.9.3.75', port=30028): Read timed out. (read timeout=600)


In [ ]:
# ═══════════════════════════════════════════════════
# PHA 2: Chunk các văn bản THAM CHIẾU (depth=1) — KHÔNG đệ quy thêm
# ═══════════════════════════════════════════════════
print("\n" + "═" * 60)
print("PHA 2: Chunking văn bản THAM CHIẾU (depth=1) (BATCH 10)")
print("═" * 60)

depth1_count = 0
disco_ids_list = list(discovered_depth1_ids)
for i in range(0, len(disco_ids_list), BATCH_SIZE):
    batch_ids = disco_ids_list[i : i + BATCH_SIZE]
    batch_docs = []
    batch_meta_map = {}
    
    for doc_id in batch_ids:
        if doc_id in processed_ids: continue
        
        idx_content = content_id_to_idx.get(doc_id)
        idx_meta = meta_id_to_idx.get(doc_id)
        if idx_content is None or idx_meta is None:
            processed_ids.add(doc_id)
            continue
        
        content = ds_content_all[idx_content].get("content", "")
        meta = all_meta_records[idx_meta]
        if not content: 
            processed_ids.add(doc_id)
            continue
            
        doc_num = str(meta.get("document_number", ""))
        batch_docs.append({"source_doc": doc_num, "content": content})
        batch_meta_map[doc_num] = {"id": doc_id, "meta": meta, "content": content}

    if not batch_docs: continue

    # BƯỚC 1: TRÍCH XUẤT RELATION THEO MẺ 10 (Cell 3 & 4)
    print(f"  [BATCH] Đang trích xuất quan hệ cho mẻ {len(batch_docs)} văn bản...")
    
    # Bọc Try-Catch để đảm bảo pipeline không bao giờ chết
    try:
        rels_map = rel.extract_ontology_relationships_batch(batch_docs, global_doc_lookup=global_doc_lookup)
    except Exception as e:
        print(f"  ⚠️ LỖI BATCH: Bỏ qua trích xuất quan hệ mẻ này do lỗi LLM - {e}")
        rels_map = {} # Dự phòng rỗng để vẫn chunk được văn bản gốc
    
    for doc_info in batch_docs:
        s_doc = doc_info["source_doc"]
        doc_data = batch_meta_map[s_doc]
        doc_id = doc_data["id"]
        meta = doc_data["meta"]
        content = doc_data["content"]
        
        precomputed = rels_map.get(s_doc, [])
        
        key = normalize_doc_key(s_doc)
        if key: meta_by_docnum_lookup[key] = meta
        
        doc_chunks = chunker.process_document(
            content=content,
            metadata=meta,
            global_doc_lookup=global_doc_lookup,
            precomputed_rels=precomputed
        )
        all_chunks.extend(doc_chunks)
        processed_ids.add(doc_id)
        depth1_count += 1

    print(f"  Đã hoàn thành mẻ tham chiếu {i//BATCH_SIZE + 1}. Tổng xử lý: {depth1_count}/{len(disco_ids_list)}")

print(f"\n✅ PHA 2 hoàn tất: Thêm {depth1_count} văn bản tham chiếu.")
print(f"🎯 TỔNG KẾT: {len(processed_ids)} văn bản ({len(target_ids)} gốc + {depth1_count} tham chiếu) → {len(all_chunks)} chunks.")



════════════════════════════════════════════════════════════
PHA 2: Chunking văn bản THAM CHIẾU (depth=1) (BATCH 10)
════════════════════════════════════════════════════════════

✅ PHA 2 hoàn tất: Thêm 0 văn bản tham chiếu.
🎯 TỔNG KẾT: 100 văn bản (100 gốc + 0 tham chiếu) → 3192 chunks.


In [ ]:
import uuid
from backend.retrieval.embedder import embedder
from backend.retrieval.vector_db import client as qdrant, ensure_collection
from backend.retrieval.graph_db import get_neo4j_driver, build_neo4j
from qdrant_client.models import PointStruct
from backend.config import settings

collection_name = "legal_hybrid_rag_docs"
# Đảm bảo Collection tồn tại trước khi thao tác
ensure_collection(qdrant, collection_name=collection_name)

# 1. Tính toán Dense và Sparse Vectors (Batching để tránh Timeout)
print("-> Bắt đầu trích xuất Vectors (Batching = 50)...")
texts_to_embed = [c.get("text_to_embed", c.get("chunk_text", "")) for c in all_chunks]

dense_vectors = []
sparse_vectors = []
batch_size = 50

for i in range(0, len(texts_to_embed), batch_size):
    batch_texts = texts_to_embed[i:i + batch_size]
    # encode_dense
    dense_batch = embedder.encode_dense(batch_texts)
    dense_vectors.extend(dense_batch)
    
    # encode_sparse_documents (để sinh SparseVector placeholder)
    sparse_batch = embedder.encode_sparse_documents(batch_texts)
    sparse_vectors.extend(sparse_batch)
    
    print(f"   Đã nhúng {min(i + batch_size, len(texts_to_embed))}/{len(texts_to_embed)} chunks...")

# 2. Đẩy Qdrant (Batching để chống Sập/OOM Docker)
print("-> Đang đẩy vào Qdrant (Batching = 200)...")
points = []
for idx, chunk in enumerate(all_chunks):
    vec_payload = {
        "dense": dense_vectors[idx],
        "sparse": sparse_vectors[idx]
    }
    q_payload = chunk.get("qdrant_metadata", chunk.get("metadata", {}))
    points.append(PointStruct(
        id=str(uuid.uuid5(uuid.NAMESPACE_DNS, chunk["chunk_id"])),
        vector=vec_payload,
        payload=q_payload
    ))

upsert_batch_size = 200
for i in range(0, len(points), upsert_batch_size):
    batch_points = points[i:i + upsert_batch_size]
    qdrant.upsert(collection_name=collection_name, points=batch_points)
    print(f"   Đã đẩy Qdrant {min(i + upsert_batch_size, len(points))}/{len(points)} chunks...")

print(f"✅ Đã tải thành công lên Local Qdrant (Collection: {collection_name}).")

# 3. Đẩy Graph Neo4j
print("-> Đang xây dựng Đồ thị tri thức (Neo4j)...")
driver = get_neo4j_driver()
if driver:
    build_neo4j(driver, all_chunks, meta_by_docnum_lookup=meta_by_docnum_lookup)
    driver.close()
    print("✅ Cập nhật Graph Neo4j hoàn thành.")
else:
    print("❌ Lỗi kết nối Neo4j.")

-> Bắt đầu trích xuất Vectors (Batching = 50)...
   Đã nhúng 50/3192 chunks...
   Đã nhúng 100/3192 chunks...
   Đã nhúng 150/3192 chunks...
   Đã nhúng 200/3192 chunks...
   Đã nhúng 250/3192 chunks...
   Đã nhúng 300/3192 chunks...
   Đã nhúng 350/3192 chunks...
   Đã nhúng 400/3192 chunks...
   Đã nhúng 450/3192 chunks...
   Đã nhúng 500/3192 chunks...
   Đã nhúng 550/3192 chunks...
   Đã nhúng 600/3192 chunks...
   Đã nhúng 650/3192 chunks...
   Đã nhúng 700/3192 chunks...
   Đã nhúng 750/3192 chunks...
   Đã nhúng 800/3192 chunks...
   Đã nhúng 850/3192 chunks...
   Đã nhúng 900/3192 chunks...
   Đã nhúng 950/3192 chunks...
   Đã nhúng 1000/3192 chunks...
   Đã nhúng 1050/3192 chunks...
   Đã nhúng 1100/3192 chunks...
   Đã nhúng 1150/3192 chunks...
   Đã nhúng 1200/3192 chunks...


In [ ]:
import sys
import os
import json

# Fix đường dẫn: Đặt working directory về thư mục gốc để script chạy đúng
ROOT_DIR = "d:/iCOMM/Legal-RAG"
os.chdir(ROOT_DIR)
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

from backend.agent.graph import LegalRAGWorkflow
from backend.llm.factory import chat_completion
from backend.config import settings

# 1. Cấu hình Logger để vừa in ra màn hình vừa ghi file
class Logger(object):
    def __init__(self, filename="results/test_chatbot_results.txt"):
        self.terminal = sys.stdout
        # Đảm bảo thư mục kết quả tồn tại
        os.makedirs(os.path.dirname(os.path.abspath(filename)), exist_ok=True)
        self.log = open(filename, "w", encoding="utf-8")

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        self.terminal.flush()
        self.log.flush()

# Đảm bảo encoding chuẩn cho terminal
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# Chuyển hướng stdout vào file
sys.stdout = Logger("results/test_chatbot_results.txt")

# 2. Khởi tạo RAG Graph
workflow = LegalRAGWorkflow()
graph = workflow.build()

# 3. Chạy test từ file Chatbot_test.json
test_cases = []
try:
    with open("Chatbot_test.json", "r", encoding="utf-8") as f:
        test_data = json.load(f)
        for item in test_data:
            test_cases.append((item["id"], item["question"], item["category"], item["answer"]))
except Exception as e:
    print(f"Lỗi đọc file json: {e}")

print("STARTING TEST: 3 RAG MODES (ON-PREMISE)\n" + "="*60)
print(f"Logging to: {os.path.abspath('results/test_chatbot_results.txt')}")

# 4. Hàm đánh giá bằng LLM Judge
def evaluate_answer(question, expected_answer, generated_answer):
    prompt = f"""Bạn là một Thẩm phán AI đánh giá câu trả lời của hệ thống RAG.
Nhiệm vụ: So sánh CÂU TRẢ LỜI CỦA RAG với CÂU TRẢ LỜI MẪU xem có chính xác và đầy đủ các ý chính hay không.

Câu hỏi: {question}
Câu trả lời mẫu: {expected_answer}
Câu trả lời từ RAG: {generated_answer}

Bạn PHẢI trả về đúng MỘT DÒNG duy nhất bắt đầu bằng "✅ ĐẠT" hoặc "❌ KHÔNG ĐẠT", kèm theo một câu giải thích cực kỳ ngắn gọn.
"""
    try:
        from backend.config import settings
        res = chat_completion([{"role": "user", "content": prompt}], temperature=0.0, model=settings.LLM_ROUTING_MODEL)
        return str(res).strip()
    except Exception as e:
        return f"❌ LỖI ĐÁNH GIÁ LLM: {str(e)}"

# 5. Vòng lặp thực thi test duy nhất
import requests

for mode_name, question, intent, expected_answer in test_cases:
    print(f"\n--- [{mode_name.upper()}] Query: {question}")
    
    initial_state = {
        "query": question, 
        "mode": "AUTO",
        "session_id": f"test_{intent}",
        "history": [],
        "use_grading": True,      
        "use_reflection": False,    
        "use_rerank": True,
        "top_k": 5
    }
    
    try:
        # invoke là synchronous call
        res = graph.invoke(initial_state)
        answer = res.get('answer') or res.get('final_response') or res.get('draft_response') or "[No Answer]"
        print(f"\n[Generated Answer]:\n{answer}")
        
        print(f"\n[Expected Answer]: {expected_answer}")
        judge_result = evaluate_answer(question, expected_answer, answer)
        print(f"\n[LLM JUDGE]: {judge_result}")
        
    except requests.exceptions.Timeout:
        print("❌ LỖI: LLM API Timeout (Phản hồi quá lâu). Đang bỏ qua câu này...")
    except requests.exceptions.RequestException as e:
        print(f"❌ LỖI MẠNG/API: {e}")
    except Exception as e:
        print(f"❌ FAILED Execution: {e}")
    
    print("-" * 60)

print("\n" + "="*60 + "\nTEST COMPLETED.")

KeyboardInterrupt: 